In [4]:
# Import required libraries
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
import warnings
import glob
import os
import collections.abc
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration: File Paths ---
DATA_DIR = "."
PATH_BANK_HOLIDAYS = os.path.join(DATA_DIR, "uk_bank_holidays.csv")
PATH_INFO_HOUSEHOLDS = os.path.join(DATA_DIR, "informations_households.csv")
PATH_WEATHER = os.path.join(DATA_DIR, "weather_daily_darksky.csv")
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, "daily_merged_predictions_tree_rules.csv")
MF_PLOT_DIR = "mf_plots"
DIST_PLOT_DIR = "distribution_plots"
DAILY_DATA_FOLDER = "daily_dataset"
DAILY_DATA_PATTERN = os.path.join(DATA_DIR, DAILY_DATA_FOLDER, "block_*.csv")

# --- Configuration: Features and Columns ---
WEATHER_FEATURES = [
    "temperatureMax", "temperatureMin", "dewPoint", "humidity",
    "cloudCover", "windSpeed", "pressure"]
DAILY_DATASET_NAN_DROP_COLS = ['energy_median', 'energy_mean', 'energy_max', 'energy_sum']
WEATHER_NAN_DROP_COLS = WEATHER_FEATURES

# --- Configuration: Plotting ---
SCATTER_PLOT_SAMPLE_SIZE = 5000
LINE_PLOT_DAYS = 90

# --- Configuration: Rule Generation ---
TREE_MAX_DEPTH = 5

# =============================================================================
# Data Loading and Preprocessing Functions
# =============================================================================
# (load_data, handle_missing_values, process_dates_holidays, aggregate_and_merge functions remain the same as previous version)
def load_data():
    """Loads all necessary datasets."""
    print("Loading datasets...")
    try:
        df_bank_holidays = pd.read_csv(PATH_BANK_HOLIDAYS)
        df_info_households = pd.read_csv(PATH_INFO_HOUSEHOLDS) # Load but don't merge later
        df_weather = pd.read_csv(PATH_WEATHER)
        print(f"Looking for daily energy files matching: {DAILY_DATA_PATTERN}")
        block_files = glob.glob(DAILY_DATA_PATTERN)
        if not block_files: print(f"Error: No files found matching the pattern '{DAILY_DATA_PATTERN}'."); exit()
        print(f"Found {len(block_files)} daily energy block files. Loading and concatenating...")
        all_daily_data = [pd.read_csv(f) for f in tqdm(block_files, desc="Loading block files")]
        all_daily_data = [df for df in all_daily_data if not df.empty and all(col in df.columns for col in ['LCLid', 'day'] + DAILY_DATASET_NAN_DROP_COLS)]
        if not all_daily_data: print("Error: No valid daily block files could be loaded or processed. Exiting."); exit()
        df_daily_dataset = pd.concat(all_daily_data, ignore_index=True)
        print(f"Combined daily dataset loaded successfully ({len(df_daily_dataset)} rows).")
        print("Core datasets loaded successfully.")
        return df_daily_dataset, df_bank_holidays, df_info_households, df_weather
    except FileNotFoundError as e: print(f"Error loading file: {e}."); exit()
    except Exception as e: print(f"An unexpected error occurred during file loading: {e}"); exit()

def handle_missing_values(df_daily_dataset, df_weather):
    """Handles missing values in daily and weather dataframes."""
    global WEATHER_NAN_DROP_COLS, WEATHER_FEATURES
    print("\nHandling missing values...")
    initial_daily_rows = len(df_daily_dataset)
    df_daily_dataset.dropna(subset=DAILY_DATASET_NAN_DROP_COLS, inplace=True)
    print(f"Dropped {initial_daily_rows - len(df_daily_dataset)} rows with missing energy data from original daily data.")
    initial_weather_rows = len(df_weather)
    if 'cloudCover' not in df_weather.columns:
        print("Warning: 'cloudCover' column not found in weather data. Removing from NaN check.")
        WEATHER_NAN_DROP_COLS = [col for col in WEATHER_NAN_DROP_COLS if col != 'cloudCover']
        WEATHER_FEATURES = [col for col in WEATHER_FEATURES if col != 'cloudCover']
    # *** DewPoint Check Added ***
    if 'dewPoint' not in df_weather.columns:
        print("Warning: 'dewPoint' column not found in weather data. Removing from NaN check.")
        WEATHER_NAN_DROP_COLS = [col for col in WEATHER_NAN_DROP_COLS if col != 'dewPoint']
        WEATHER_FEATURES = [col for col in WEATHER_FEATURES if col != 'dewPoint']
    # **************************
    valid_weather_nan_drop_cols = [col for col in WEATHER_NAN_DROP_COLS if col in df_weather.columns]
    if len(valid_weather_nan_drop_cols) < len(WEATHER_NAN_DROP_COLS):
        print(f"Warning: Some weather columns specified for NaN drop were not found: {set(WEATHER_NAN_DROP_COLS) - set(valid_weather_nan_drop_cols)}")
    if valid_weather_nan_drop_cols:
        df_weather.dropna(subset=valid_weather_nan_drop_cols, inplace=True)
        print(f"Dropped {initial_weather_rows - len(df_weather)} rows with missing weather data.")
    else:
        print("Skipping weather NaN drop as no valid key columns were found.")
    return df_daily_dataset, df_weather

def process_dates_holidays(df_daily_dataset, df_weather, df_bank_holidays):
    """Converts date columns and prepares holiday flags."""
    print("\nProcessing dates and holidays...")
    if 'time' in df_weather.columns and pd.api.types.is_numeric_dtype(df_weather['time']):
        df_weather['time'] = pd.to_datetime(df_weather['time'], unit='s', errors='coerce')
    elif 'time' in df_weather.columns:
         df_weather['time'] = pd.to_datetime(df_weather['time'], errors='coerce')
    else: print("Warning: 'time' column not found in weather data.")
    df_weather.dropna(subset=['time'], inplace=True)
    df_weather['day'] = df_weather['time'].dt.normalize()
    df_daily_dataset['day'] = pd.to_datetime(df_daily_dataset['day'], errors='coerce')
    df_daily_dataset.dropna(subset=['day'], inplace=True)
    holiday_col_name = 'Bank holidays'
    if holiday_col_name not in df_bank_holidays.columns:
        potential_date_cols = [col for col in df_bank_holidays.columns if 'date' in col.lower()]
        if potential_date_cols: holiday_col_name = potential_date_cols[0]
        else: print(f"Error: Date column not found in {PATH_BANK_HOLIDAYS}."); exit()
    df_bank_holidays.rename(columns={holiday_col_name: 'day'}, inplace=True)
    try: df_bank_holidays['day'] = pd.to_datetime(df_bank_holidays['day'], format='%Y-%m-%d', errors='coerce')
    except ValueError: df_bank_holidays['day'] = pd.to_datetime(df_bank_holidays['day'], errors='coerce')
    df_bank_holidays.dropna(subset=['day'], inplace=True)
    df_bank_holidays['is_holiday'] = 1
    df_bank_holidays = df_bank_holidays[['day', 'is_holiday']].drop_duplicates(subset='day')
    print("Date conversions complete.")
    return df_daily_dataset, df_weather, df_bank_holidays

def aggregate_and_merge(df_daily_dataset, df_weather, df_bank_holidays):
    """Calculates daily average energy and merges with weather/holidays."""
    print("\nAggregating daily energy and merging datasets...")
    print("Calculating daily average energy per household...")
    df_daily_agg = df_daily_dataset.groupby('day').agg(
        total_energy_sum=('energy_sum', 'sum'),
        household_count=('LCLid', 'nunique')
    ).reset_index()
    df_daily_agg['avg_energy_per_household'] = df_daily_agg['total_energy_sum'] / df_daily_agg['household_count']
    df_daily_agg['avg_energy_per_household'].replace([np.inf, -np.inf], np.nan, inplace=True)
    df_daily_agg.dropna(subset=['avg_energy_per_household'], inplace=True)
    print(f"Daily average energy calculated for {len(df_daily_agg)} days.")
    weather_cols_to_merge = ['day'] + [col for col in WEATHER_FEATURES if col in df_weather.columns]
    df_weather_daily = df_weather[weather_cols_to_merge].drop_duplicates(subset='day', keep='first')
    print("Merging daily aggregated energy with weather and holidays...")
    df_merged = pd.merge(df_daily_agg, df_weather_daily, on='day', how='inner')
    print(f"Rows after merging aggregated energy with weather: {len(df_merged)}")
    df_merged = pd.merge(df_merged, df_bank_holidays, on='day', how='left')
    df_merged['is_holiday'] = df_merged['is_holiday'].fillna(0).astype(int)
    print(f"Rows after adding holidays: {len(df_merged)}")
    df_merged['month'] = df_merged['day'].dt.month
    print("Added 'month' column.")
    if df_merged.empty: print("\nError: DataFrame empty after merging."); exit()
    print(f"\nTotal rows in final merged DataFrame (Daily Level): {len(df_merged)}")
    print("Columns:", df_merged.columns.tolist())
    return df_merged

def normalize_features(df_merged):
    """Normalizes weather features and actual average energy, creates lagged feature."""
    print("\nNormalizing features...")
    # Normalize weather features
    weather_cols_to_normalize = [col for col in WEATHER_FEATURES if col in df_merged.columns]
    print(f"Weather columns to normalize: {weather_cols_to_normalize}")
    if weather_cols_to_normalize:
        for col in weather_cols_to_normalize:
            df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')
        weather_cols_to_normalize = [col for col in weather_cols_to_normalize if col in df_merged.columns and pd.api.types.is_numeric_dtype(df_merged[col])]
        print(f"Actual weather columns being normalized: {weather_cols_to_normalize}")
        medians = df_merged[weather_cols_to_normalize].median()
        for col in weather_cols_to_normalize:
            if df_merged[col].isnull().any():
                 if col in medians and pd.notna(medians[col]): df_merged[col].fillna(medians[col], inplace=True)
                 else: print(f"Warning: Could not fill NaNs in '{col}'.")
        nan_check = df_merged[weather_cols_to_normalize].isnull().sum().sum()
        inf_check = np.isinf(df_merged[weather_cols_to_normalize]).sum().sum()
        if nan_check > 0 or inf_check > 0:
            print(f"ERROR: {nan_check} NaNs / {inf_check} Infs detected before scaling weather! Dropping rows.")
            df_merged.dropna(subset=weather_cols_to_normalize, inplace=True)
        weather_scaler = MinMaxScaler()
        df_merged[weather_cols_to_normalize] = weather_scaler.fit_transform(df_merged[weather_cols_to_normalize])
        nan_check_after = df_merged[weather_cols_to_normalize].isnull().sum().sum()
        inf_check_after = np.isinf(df_merged[weather_cols_to_normalize]).sum().sum()
        if nan_check_after > 0 or inf_check_after > 0:
            print(f"ERROR: {nan_check_after} NaNs / {inf_check_after} Infs detected after scaling weather! Dropping rows.")
            df_merged.dropna(subset=weather_cols_to_normalize, inplace=True)
        print("Weather feature normalization complete.")
    else: print("Skipping weather normalization as no valid numeric columns were found.")

    # Normalize actual average energy and create lagged feature
    print("Normalizing actual average energy per household for comparison...")
    actual_energy_col = 'avg_energy_per_household'
    normalized_col = 'actual_energy_normalized'
    lagged_col = 'previous_day_energy_normalized'
    if actual_energy_col in df_merged.columns:
        df_merged[actual_energy_col] = pd.to_numeric(df_merged[actual_energy_col], errors='coerce')
        if df_merged[actual_energy_col].isnull().any():
            actual_median = df_merged[actual_energy_col].median()
            print(f"Filling {df_merged[actual_energy_col].isnull().sum()} NaNs in '{actual_energy_col}' with median ({actual_median}).")
            df_merged[actual_energy_col].fillna(actual_median, inplace=True)
        inf_check_actual = np.isinf(df_merged[actual_energy_col]).sum()
        if inf_check_actual > 0:
            print(f"ERROR: Found {inf_check_actual} infinite values in '{actual_energy_col}'. Replacing.")
            df_merged[actual_energy_col].replace([np.inf, -np.inf], np.nan, inplace=True)
            actual_median = df_merged[actual_energy_col].median()
            df_merged[actual_energy_col].fillna(actual_median, inplace=True)
        actual_energy_scaler = MinMaxScaler()
        actual_energy_values = df_merged[actual_energy_col].values.reshape(-1, 1)
        df_merged[normalized_col] = actual_energy_scaler.fit_transform(actual_energy_values)
        print(f"'{actual_energy_col}' normalized and stored in '{normalized_col}'.")

        # --- Create Lagged Feature using bfill ---
        df_merged[lagged_col] = df_merged[normalized_col].shift(1)
        df_merged[lagged_col].fillna(method='bfill', limit=1, inplace=True)
        print(f"Created '{lagged_col}' feature and filled initial NaN using backward fill.")
        if df_merged[lagged_col].isnull().any():
             mean_normalized_energy = df_merged[normalized_col].mean()
             df_merged[lagged_col].fillna(mean_normalized_energy, inplace=True)
             print(f"Filled remaining NaNs in '{lagged_col}' with mean ({mean_normalized_energy:.4f}).")

    else:
        print(f"Warning: Column '{actual_energy_col}' not found. Cannot normalize or create lagged feature.")
        df_merged[normalized_col] = np.nan
        df_merged[lagged_col] = np.nan

    return df_merged, weather_cols_to_normalize

# =============================================================================
# Fuzzy System Definition and Prediction Functions
# =============================================================================

def define_fuzzy_variables_and_mfs():
    """Defines all fuzzy antecedents and consequents with their MFs."""
    print("\nDefining fuzzy variables and membership functions...")
    # Universes
    norm_universe = np.arange(-0.01, 1.02, 0.01) # For normalized features [0, 1]
    month_universe = np.arange(0.5, 13.5, 0.5)   # For month (1-12)
    holiday_universe = np.arange(-0.1, 1.1, 0.1) # For holiday flag (0 or 1)

    antecedents = {}
    # Weather Antecedents
    weather_cols = [col for col in WEATHER_FEATURES if col in ["temperatureMax", "temperatureMin", "humidity", "cloudCover", "windSpeed", "pressure"]]
    for col in weather_cols:
         antecedents[col] = ctrl.Antecedent(norm_universe, col)
    # Time Antecedents
    antecedents['Month'] = ctrl.Antecedent(month_universe, 'Month')
    antecedents['Holiday'] = ctrl.Antecedent(holiday_universe, 'Holiday')
    # Lagged Energy Antecedent
    antecedents['Previous_Energy'] = ctrl.Antecedent(norm_universe, 'Previous_Energy')

    # Consequent
    predictedEnergy = ctrl.Consequent(norm_universe, 'predictedEnergy')
    print(f"Fuzzy antecedents defined: {list(antecedents.keys())}")

    # --- Membership Functions Definitions ---
    # Temperature MFs based on histograms
    temp_labels = ['very_cold', 'cold', 'cool', 'warm', 'hot']
    temp_mfs = { 'very_cold': fuzz.trimf(norm_universe, [-0.01, 0.15, 0.3]), 'cold': fuzz.trimf(norm_universe, [0.2, 0.35, 0.5]), 'cool': fuzz.trimf(norm_universe, [0.4, 0.55, 0.7]), 'warm': fuzz.trimf(norm_universe, [0.6, 0.75, 0.9]), 'hot': fuzz.trimf(norm_universe, [0.8, 0.95, 1.01]) }
    # Humidity MFs based on histogram
    hum_labels = ['dry', 'comfortable', 'humid']
    hum_mfs = { 'dry': fuzz.trapmf(norm_universe, [0, 0, 0.2, 0.4]), 'comfortable': fuzz.trapmf(norm_universe, [0.3, 0.5, 0.7, 0.8]), 'humid': fuzz.trapmf(norm_universe, [0.7, 0.85, 1, 1]) }
    low_med_high_labels = ['low', 'medium', 'high']
    # CloudCover MFs based on histogram
    cloud_cover_mfs = { 'low': fuzz.trimf(norm_universe, [-0.01, 0.2, 0.5]), 'medium': fuzz.trimf(norm_universe, [0.3, 0.55, 0.8]), 'high': fuzz.trimf(norm_universe, [0.65, 0.85, 1.01]) }
    # Pressure MFs based on histogram
    pressure_mfs = { 'low': fuzz.trimf(norm_universe, [-0.01, 0.3, 0.55]), 'medium': fuzz.trimf(norm_universe, [0.45, 0.6, 0.75]), 'high': fuzz.trimf(norm_universe, [0.65, 0.85, 1.01]) }
    # WindSpeed MFs based on histogram
    wind_mfs = { 'low': fuzz.trimf(norm_universe, [-0.01, 0.25, 0.45]), 'medium': fuzz.trimf(norm_universe, [0.35, 0.5, 0.7]), 'high': fuzz.trimf(norm_universe, [0.6, 0.8, 1.01]) }
    # *** MODIFIED Previous_Energy MFs based on histogram ***
    lag_mfs = { 'low': fuzz.trimf(norm_universe, [0.3, 0.45, 0.6]), 'medium': fuzz.trimf(norm_universe, [0.5, 0.65, 0.8]), 'high': fuzz.trimf(norm_universe, [0.7, 0.85, 1.01]) }
    # ----------------------------------------------------
    # Month (4 seasons)
    season_labels = ['Winter', 'Spring', 'Summer', 'Autumn']
    season_mfs = { 'Winter': fuzz.trapmf(month_universe, [0.5, 0.5, 2.5, 3.5]), 'Spring': fuzz.trapmf(month_universe, [2.5, 3.5, 5.5, 6.5]), 'Summer': fuzz.trapmf(month_universe, [5.5, 6.5, 8.5, 9.5]), 'Autumn': fuzz.trapmf(month_universe, [8.5, 9.5, 11.5, 12.5]) }
    # Holiday (2 levels)
    holiday_labels = ['No', 'Yes']
    holiday_mfs = { 'No': fuzz.trimf(holiday_universe, [-0.1, 0, 0.1]), 'Yes': fuzz.trimf(holiday_universe, [0.9, 1, 1.1]) }
    # Predicted Energy MFs based on actual energy distribution
    energy_labels = ['Low', 'Medium', 'High', 'Very_High']
    energy_mfs = { 'Low': fuzz.trimf(norm_universe, [-0.01, 0.2, 0.4]), 'Medium': fuzz.trimf(norm_universe, [0.3, 0.55, 0.7]), 'High': fuzz.trimf(norm_universe, [0.6, 0.75, 0.9]), 'Very_High': fuzz.trimf(norm_universe, [0.8, 0.95, 1.01]) }

    # Apply MFs to antecedents
    for name, var in antecedents.items():
        if 'temperature' in name:
            for label in temp_labels: var[label] = temp_mfs[label]
        elif 'humidity' in name:
            for label in hum_labels: var[label] = hum_mfs[label]
        elif name == 'cloudCover':
             for label in low_med_high_labels: var[label] = cloud_cover_mfs[label]
        elif name == 'pressure':
             for label in low_med_high_labels: var[label] = pressure_mfs[label]
        elif name == 'windSpeed':
             for label in low_med_high_labels: var[label] = wind_mfs[label]
        elif name == 'Previous_Energy': # *** Apply specific MFs for Previous_Energy ***
             for label in low_med_high_labels: var[label] = lag_mfs[label]
        # ----------------------------------------------------
        elif name == 'dewPoint': # Apply generic low/med/high to dewPoint
            generic_lmh = { 'low': fuzz.trimf(norm_universe, [-0.01, 0.15, 0.45]), 'medium': fuzz.trimf(norm_universe, [0.15, 0.5, 0.85]), 'high': fuzz.trimf(norm_universe, [0.55, 0.85, 1.01]) }
            print(f"Assigning generic low/med/high MFs to dewPoint")
            for label in low_med_high_labels: var[label] = generic_lmh[label]
        elif name == 'Month':
             for label in season_labels: var[label] = season_mfs[label]
        elif name == 'Holiday':
             for label in holiday_labels: var[label] = holiday_mfs[label]

    # Apply MFs to the consequent
    for label in energy_labels:
        predictedEnergy[label] = energy_mfs[label]

    print("Membership functions defined.")
    return antecedents, predictedEnergy

def visualize_membership_functions(antecedents, consequent):
    """Generates and saves plots for all membership functions."""
    print("\nGenerating Membership Function plots...")
    if not os.path.exists(MF_PLOT_DIR): os.makedirs(MF_PLOT_DIR)
    for name, var in antecedents.items():
        try:
            var.view()
            plot_filename = os.path.join(MF_PLOT_DIR, f"mf_{name}.png")
            plt.savefig(plot_filename); plt.close()
            print(f"Saved MF plot for {name} to {plot_filename}")
        except Exception as e: print(f"Could not plot MF for {name}: {e}")
    try:
        consequent.view()
        plot_filename = os.path.join(MF_PLOT_DIR, f"mf_{consequent.label}.png")
        plt.savefig(plot_filename); plt.close()
        print(f"Saved MF plot for {consequent.label} to {plot_filename}")
    except Exception as e: print(f"Could not plot MF for {consequent.label}: {e}")
    print("Finished generating MF plots.")

# *** NEW FUNCTION: Visualize Data Distributions ***
def visualize_data_distributions(df, normalized_weather_cols):
    """Generates and saves histograms for normalized features."""
    print("\nGenerating Data Distribution plots...")
    if not os.path.exists(DIST_PLOT_DIR): os.makedirs(DIST_PLOT_DIR)

    # Columns to plot histograms for
    cols_to_plot = normalized_weather_cols + ['previous_day_energy_normalized', 'actual_energy_normalized']

    for col in cols_to_plot:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            try:
                plt.figure(figsize=(8, 5))
                sns.histplot(df[col].dropna(), kde=True, bins=30) # Use dropna()
                plt.title(f'Distribution of Normalized {col}')
                plt.xlabel('Normalized Value')
                plt.ylabel('Frequency')
                plt.grid(True, axis='y')
                plot_filename = os.path.join(DIST_PLOT_DIR, f"dist_{col}.png")
                plt.savefig(plot_filename)
                plt.close()
                print(f"Saved distribution plot for {col} to {plot_filename}")
            except Exception as e:
                print(f"Could not plot distribution for {col}: {e}")
        else:
            print(f"Skipping distribution plot for {col} (not found or not numeric).")
    print("Finished generating distribution plots.")


def fuzzify_to_labels(df, antecedents, consequent):
    """Adds columns with the crisp fuzzy label for each variable."""
    print("\nAssigning crisp fuzzy labels based on max membership...")
    df_labels = df.copy()

    # Fuzzify antecedents
    for name, var in antecedents.items():
        label_col = f"{name}_label"
        # Find the label with the max membership degree for each row
        labels = list(var.terms.keys())
        input_col = ''
        if name == 'Month': input_col = 'month'
        elif name == 'Holiday': input_col = 'is_holiday'
        elif name == 'Previous_Energy': input_col = 'previous_day_energy_normalized'
        else: input_col = name # Weather features

        if input_col not in df.columns:
            print(f"Warning: Input column '{input_col}' for antecedent '{name}' not found. Skipping labeling.")
            continue
        # Ensure input column is numeric before interp_membership
        if not pd.api.types.is_numeric_dtype(df_labels[input_col]):
            print(f"Warning: Input column '{input_col}' is not numeric. Skipping labeling for {name}.")
            continue


        # Calculate membership for all labels efficiently
        # Handle potential NaNs in input column before fuzzification
        input_values = df_labels[input_col].fillna(var.universe.mean()).values # Fill NaNs with mean of universe
        memberships = np.array([fuzz.interp_membership(var.universe, var[label].mf, input_values) for label in labels])
        # Find the index (and thus label) of the max membership for each row
        max_indices = np.argmax(memberships, axis=0)
        df_labels[label_col] = [labels[i] for i in max_indices]
        print(f"Generated labels for {name}.")

    # Fuzzify the actual energy output
    label_col = "actual_energy_label"
    labels = list(consequent.terms.keys())
    input_col = 'actual_energy_normalized'
    if input_col in df.columns:
        if not pd.api.types.is_numeric_dtype(df_labels[input_col]):
             print(f"Warning: Input column '{input_col}' is not numeric. Skipping actual energy labeling.")
        else:
            input_values = df_labels[input_col].fillna(consequent.universe.mean()).values
            memberships = np.array([fuzz.interp_membership(consequent.universe, consequent[label].mf, input_values) for label in labels])
            max_indices = np.argmax(memberships, axis=0)
            df_labels[label_col] = [labels[i] for i in max_indices]
            print(f"Generated labels for actual energy.")
    else:
        print(f"Warning: Column '{input_col}' not found. Skipping actual energy labeling.")

    return df_labels

def generate_rules_from_tree(df_labels, antecedents):
    """Trains a Decision Tree on labels and prints extracted rules."""
    print("\nGenerating rules from Decision Tree...")
    antecedent_label_cols = [f"{name}_label" for name in antecedents.keys() if f"{name}_label" in df_labels.columns]
    target_col = "actual_energy_label"

    if not antecedent_label_cols or target_col not in df_labels.columns:
        print("Error: Missing label columns for decision tree training.")
        return None # Indicate failure

    # Check if target column has valid labels
    if df_labels[target_col].isnull().any() or df_labels[target_col].nunique() < 2:
        print(f"Error: Target column '{target_col}' has missing values or less than 2 unique labels. Cannot train tree.")
        return None

    X = df_labels[antecedent_label_cols]
    y = df_labels[target_col]

    # Check value counts before splitting
    print("\nValue counts for target variable 'actual_energy_label' before split:")
    print(y.value_counts())

    # One-hot encode the label columns for the tree
    try:
        X_encoded = pd.get_dummies(X)
    except Exception as e:
        print(f"Error during one-hot encoding: {e}")
        return None

    # Split data
    try:
        # *** MODIFIED: Removed stratify=y to fix ValueError ***
        X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42)
        print("Data split successfully.")
    except ValueError as e:
        print(f"ValueError during train_test_split: {e}")
        print("Check the distribution of your target variable 'actual_energy_label'.")
        print(y.value_counts())
        return None
    except Exception as e:
        print(f"Unexpected error during train_test_split: {e}")
        return None


    # Train the Decision Tree
    dtc = DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH, random_state=42, min_samples_leaf=10) # Limit depth and leaf size
    dtc.fit(X_train, y_train)

    print(f"Decision Tree Accuracy on test set: {dtc.score(X_test, y_test):.4f}")

    # Extract and print text rules
    print("\n--- Extracted Decision Tree Rules ---")
    try:
        # Use the feature names from the encoded dataframe
        tree_rules = export_text(dtc, feature_names=list(X_encoded.columns))
        print(tree_rules)
        print("------------------------------------")
        print("NOTE: Manually translate the rules above into skfuzzy format for the define_fuzzy_rules function.")
        return tree_rules # Return the text rules
    except Exception as e:
        print(f"Error exporting tree rules: {e}")
        return None

def define_fuzzy_rules(antecedents, predictedEnergy):
    """Defines the fuzzy rules based on insights (manual translation needed)."""
    # *** MODIFIED: Rules inspired by potential Tree output ***
    print("Defining fuzzy rules (inspired by Decision Tree analysis)...")
    rules = []
    antecedents_in_rules = set()

    # --- Example Rules (Replace/Refine based on printed tree_rules) ---
    # Prioritize strong predictors likely identified by the tree: Temp, Prev Energy, Month

    # Cold Winter Rules (Likely High/VeryHigh Energy)
    if 'temperatureMin' in antecedents and 'Month' in antecedents and 'Previous_Energy' in antecedents:
        rules.append(ctrl.Rule(antecedents['temperatureMin']['very_cold'] & antecedents['Month']['Winter'], predictedEnergy['Very_High']))
        rules.append(ctrl.Rule(antecedents['temperatureMin']['cold'] & antecedents['Month']['Winter'] & antecedents['Previous_Energy']['high'], predictedEnergy['Very_High']))
        rules.append(ctrl.Rule(antecedents['temperatureMin']['cold'] & antecedents['Month']['Winter'] & antecedents['Previous_Energy']['medium'], predictedEnergy['High']))
        antecedents_in_rules.update(['temperatureMin', 'Month', 'Previous_Energy'])

    # Cool/Warm Spring/Autumn Rules (Likely Medium/Low Energy)
    if 'temperatureMax' in antecedents and 'Month' in antecedents and 'Previous_Energy' in antecedents:
        rules.append(ctrl.Rule(antecedents['temperatureMax']['cool'] & (antecedents['Month']['Spring'] | antecedents['Month']['Autumn']), predictedEnergy['Low']))
        rules.append(ctrl.Rule(antecedents['temperatureMax']['warm'] & (antecedents['Month']['Spring'] | antecedents['Month']['Autumn']) & antecedents['Previous_Energy']['low'], predictedEnergy['Medium']))
        antecedents_in_rules.update(['temperatureMax', 'Month', 'Previous_Energy'])

    # Hot Summer Rules (Likely High Energy)
    if 'temperatureMax' in antecedents and 'Month' in antecedents and 'humidity' in antecedents:
        rules.append(ctrl.Rule(antecedents['temperatureMax']['hot'] & antecedents['Month']['Summer'] & antecedents['humidity']['humid'], predictedEnergy['Very_High']))
        rules.append(ctrl.Rule(antecedents['temperatureMax']['hot'] & antecedents['Month']['Summer'] & antecedents['humidity']['comfortable'], predictedEnergy['High']))
        antecedents_in_rules.update(['temperatureMax', 'Month', 'humidity'])

    # Previous Energy Influence (General)
    if 'Previous_Energy' in antecedents:
        rules.append(ctrl.Rule(antecedents['Previous_Energy']['low'], predictedEnergy['Low']))
        rules.append(ctrl.Rule(antecedents['Previous_Energy']['high'], predictedEnergy['High']))
        antecedents_in_rules.add('Previous_Energy')

    # Add a simple holiday rule
    if 'Holiday' in antecedents:
        rules.append(ctrl.Rule(antecedents['Holiday']['Yes'], predictedEnergy['Medium']))
        antecedents_in_rules.add('Holiday')

    # *** Default Rule: Output Medium ***
    # (Using Previous_Energy as default antecedent if available)
    default_antecedent_key = None
    # ... (Keep the default rule logic from the previous version) ...
    if 'Previous_Energy' in antecedents: default_antecedent_key = 'Previous_Energy'
    elif 'temperatureMax' in antecedents: default_antecedent_key = 'temperatureMax'
    elif 'humidity' in antecedents: default_antecedent_key = 'humidity'
    elif antecedents: default_antecedent_key = list(antecedents.keys())[0]

    if default_antecedent_key:
         # Determine labels based on antecedent type
         temp_labels = ['very_cold', 'cold', 'cool', 'warm', 'hot']
         hum_labels = ['dry', 'comfortable', 'humid']
         low_med_high_labels = ['low', 'medium', 'high']
         season_labels = ['Winter', 'Spring', 'Summer', 'Autumn']
         holiday_labels = ['No', 'Yes']

         if 'temperature' in default_antecedent_key: labels = temp_labels
         elif 'humidity' in default_antecedent_key: labels = hum_labels
         elif default_antecedent_key in ['windSpeed', 'pressure', 'cloudCover', 'Previous_Energy']: labels = low_med_high_labels
         elif default_antecedent_key == 'Month': labels = season_labels
         elif default_antecedent_key == 'Holiday': labels = holiday_labels
         else: labels = None

         if labels:
             always_true_condition = None
             for label in labels:
                 # Check if the label exists for the antecedent
                 if label in antecedents[default_antecedent_key].terms:
                     if always_true_condition is None:
                         always_true_condition = antecedents[default_antecedent_key][label]
                     else:
                         always_true_condition |= antecedents[default_antecedent_key][label]
                 else:
                     print(f"Warning: Label '{label}' not found for antecedent '{default_antecedent_key}' in default rule.")


             if always_true_condition is not None:
                 default_rule = ctrl.Rule(always_true_condition, predictedEnergy['Medium'])
                 rules.append(default_rule)
                 print("Added a default rule (output: Medium).")
             else:
                 print("Warning: Could not construct condition for default rule.")
         else:
             print(f"Warning: Could not determine labels for default rule antecedent '{default_antecedent_key}'.")
    else:
        print("Warning: Could not add a default rule.")


    if not rules: print("Error: No valid rules created."); exit()
    # Ensure all antecedents used in rules are added to the set
    for rule in rules:
        if hasattr(rule, 'antecedent'):
             # Extract antecedent names used in the rule (handles single & combined antecedents)
             if isinstance(rule.antecedent, fuzz.control.term.Term):
                 if hasattr(rule.antecedent, 'parent') and hasattr(rule.antecedent.parent, 'label'):
                      antecedents_in_rules.add(rule.antecedent.parent.label)
             elif isinstance(rule.antecedent, collections.abc.Iterable): # Check if it's iterable (like a tuple from & or |)
                 for term in rule.antecedent:
                     if hasattr(term, 'parent') and hasattr(term.parent, 'label'):
                          antecedents_in_rules.add(term.parent.label)

    print(f"Defined {len(rules)} fuzzy rules (Tree-Inspired) using antecedents: {antecedents_in_rules}")

    return rules, antecedents_in_rules

def create_fuzzy_simulation(rules, predictedEnergy):
    """Creates the fuzzy control system and simulation object."""
    print("Creating fuzzy control system and simulation...")
    try:
        energy_ctrl = ctrl.ControlSystem(rules=rules)
        predictedEnergy.defuzzify_method = 'centroid'
        energy_simulation = ctrl.ControlSystemSimulation(energy_ctrl)
        print("Control system and simulation created.")
        return energy_simulation
    except Exception as e:
        print(f"Error creating control system or simulation: {e}")
        exit()

def run_predictions(df_merged, energy_simulation, antecedents, predictedEnergy, antecedents_in_rules):
    """Runs fuzzy predictions on the DataFrame."""
    print("\n--- Running Manual Simulation Test ---")
    try:
        test_inputs = {key: 0.5 for key in antecedents.keys() if key not in ['Month', 'Holiday', 'Previous_Energy']}
        test_inputs['Month'] = 6; test_inputs['Holiday'] = 0; test_inputs['Previous_Energy'] = 0.5
        print(f"Manual test inputs: {test_inputs}")
        energy_simulation.reset()
        for key, value in test_inputs.items():
            try: energy_simulation.input[key] = value
            except ValueError: print(f"Warning (Manual Test): Could not set input for '{key}'. Skipping.")
            except KeyError: print(f"Warning (Manual Test): Input key '{key}' not found. Skipping.")
        energy_simulation.compute()
        manual_output = energy_simulation.output['predictedEnergy']
        print(f"Manual test output (predictedEnergy): {manual_output}")
        if pd.isna(manual_output): print("MANUAL TEST FAILED: NaN output.")
        else: print("MANUAL TEST SUCCEEDED.")
    except Exception as e: print(f"MANUAL TEST FAILED with exception: {type(e).__name__}: {e}")
    print("--- End Manual Simulation Test ---\n")

    print("Running fuzzy predictions on DataFrame...")
    prediction_errors = 0; error_log_count = 0; MAX_ERROR_LOGS = 5
    column_to_input_map = {key: key for key in antecedents.keys()}
    column_to_input_map['Month'] = 'month'
    column_to_input_map['Holiday'] = 'is_holiday'
    column_to_input_map['Previous_Energy'] = 'previous_day_energy_normalized'

    def apply_sim_row(row, sim, pred_energy_obj, antecedents_used, col_map):
        nonlocal prediction_errors, error_log_count
        sim.reset(); input_values_for_sim = {}
        try:
            for ante_key in antecedents.keys():
                 df_col = col_map.get(ante_key)
                 # Check if the antecedent is actually used in rules before setting input
                 if ante_key not in antecedents_used and ante_key not in ['Month', 'Holiday', 'Previous_Energy']: # Always set time/lagged if defined
                     continue

                 if ante_key in ['Month', 'Holiday']: input_val = row.get(df_col, 0)
                 elif ante_key == 'Previous_Energy': input_val = row.get(df_col, 0.5); input_val = 0.5 if pd.isna(input_val) else input_val
                 else: input_val = 0.5;
                 if df_col and df_col in row.index and pd.notna(row[df_col]):
                     if ante_key not in ['Month', 'Holiday', 'Previous_Energy']:
                         clipped = max(0.0, min(1.0, row[df_col])); input_val = clipped if pd.notna(clipped) else 0.5

                 try: sim.input[ante_key] = input_val; input_values_for_sim[ante_key] = input_val
                 except (ValueError, KeyError) as e:
                     # Only log error if the key was expected to be used
                     if ante_key in antecedents_used and error_log_count < MAX_ERROR_LOGS:
                          print(f"DEBUG (run_simulation): Error setting input for '{ante_key}': {e}")
                          error_log_count +=1


            sim.compute(); output_dict = sim.output
            if pred_energy_obj.label in output_dict: pred_val = output_dict[pred_energy_obj.label]
            else: prediction_errors += 1; return np.nan # Log full error if needed
            if pd.isna(pred_val): prediction_errors += 1; return np.nan # Log full error if needed
            return pred_val
        except Exception as e: prediction_errors += 1; return np.nan # Log full error if needed

    print(f"Applying simulation to {len(df_merged)} daily rows...")
    tqdm.pandas(desc="Applying fuzzy simulation")
    df_merged['predicted_energy'] = df_merged.progress_apply(
        lambda row: apply_sim_row(row, energy_simulation, predictedEnergy, antecedents_in_rules, column_to_input_map), axis=1)

    if prediction_errors > 0:
        print(f"\nWarning: Encountered {prediction_errors} errors during fuzzy simulation.")
        nan_predictions = df_merged['predicted_energy'].isnull().sum()
        print(f"Total NaN values in 'predicted_energy' column: {nan_predictions}")
        if nan_predictions == len(df_merged): print("All predictions resulted in NaN.")
        elif nan_predictions > 0:
             predicted_median = df_merged['predicted_energy'].median()
             if pd.notna(predicted_median):
                  df_merged['predicted_energy'].fillna(predicted_median, inplace=True)
                  print(f"Filled {nan_predictions} NaN predictions with median value: {predicted_median:.4f}")
             else: print("Could not calculate median for predictions. Leaving NaNs.")
    print("Predictions complete.")
    return df_merged

def perform_analysis(df_merged):
    """Performs correlation, group analysis, and plotting on daily data."""
    print("\n--- Performing Analysis ---")
    if 'predicted_energy' in df_merged.columns and \
       'actual_energy_normalized' in df_merged.columns and \
       df_merged['predicted_energy'].notna().any() and \
       df_merged['actual_energy_normalized'].notna().any():
        correlation = df_merged['predicted_energy'].corr(df_merged['actual_energy_normalized'])
        print(f"\nCorrelation between predicted_energy and actual_energy_normalized: {correlation:.4f}")
        print(f"\nGenerating scatter plot (Predicted vs Actual)...")
        try:
            plot_sample_df = df_merged
            if len(df_merged) > SCATTER_PLOT_SAMPLE_SIZE:
                print(f"(Sampling {SCATTER_PLOT_SAMPLE_SIZE} points for plot)")
                plot_sample_df = df_merged.sample(n=SCATTER_PLOT_SAMPLE_SIZE, random_state=42)
            plt.figure(figsize=(10, 6))
            sns.scatterplot(data=plot_sample_df, x='actual_energy_normalized', y='predicted_energy', alpha=0.6)
            plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Perfect Prediction')
            plt.title('Predicted Energy Score vs. Normalized Actual Avg Energy (Daily)')
            plt.xlabel('Normalized Actual Avg Energy Per Household')
            plt.ylabel('Predicted Energy (Fuzzy Score)')
            plt.grid(True); plt.legend()
            plt.savefig("scatter_plot_daily_pred_vs_actual.png"); plt.close()
            print("Scatter plot saved as scatter_plot_daily_pred_vs_actual.png.")
        except ImportError: print("Error: Matplotlib or Seaborn not installed.")
        except Exception as e: print(f"Error generating scatter plot: {e}")
        print("\nGroup Analysis by Month:")
        try:
            if 'month' in df_merged.columns:
                monthly_analysis = df_merged.groupby('month')[['actual_energy_normalized', 'predicted_energy']].mean()
                print(monthly_analysis)
            else: print("Month column not found for analysis.")
        except Exception as e: print(f"Error performing group analysis by month: {e}")
        print(f"\nGenerating line plot for daily average energy (first {LINE_PLOT_DAYS} days)...")
        try:
            plot_df = df_merged.sort_values('day').head(LINE_PLOT_DAYS)
            if not plot_df.empty:
                plt.figure(figsize=(15, 7))
                plt.plot(plot_df['day'], plot_df['actual_energy_normalized'], label='Actual Avg Energy (Normalized)', marker='.', linestyle='-', alpha=0.7)
                plt.plot(plot_df['day'], plot_df['predicted_energy'], label='Predicted Energy (Fuzzy Score)', marker='x', linestyle='--', alpha=0.7)
                plt.title(f'Daily Avg Energy vs. Prediction Score (First {LINE_PLOT_DAYS} Days)')
                plt.xlabel('Date'); plt.ylabel('Normalized Value / Fuzzy Score')
                plt.legend(); plt.grid(True); plt.xticks(rotation=45); plt.tight_layout()
                plt.savefig("line_plot_daily_time_series.png"); plt.close()
                print("Line plot saved as line_plot_daily_time_series.png.")
            else: print(f"No data found for the first {LINE_PLOT_DAYS} days.")
        except ImportError: print("Error: Matplotlib not installed.")
        except Exception as e: print(f"Error generating line plot: {e}")
    else: print("\nSkipping analysis: Required columns missing or contain only NaNs.")
    print("--- Analysis Complete ---")
    return df_merged

def inspect_and_save(df_merged):
    """Inspects final results and saves the DataFrame to CSV."""
    global OUTPUT_CSV_PATH
    print("\nInspecting final results sample (Daily Level)...")
    columns_to_show_sample = [col for col in ['day', 'avg_energy_per_household', 'temperatureMax', 'humidity', 'month', 'is_holiday', 'previous_day_energy_normalized'] if col in df_merged.columns] \
                  + ['actual_energy_normalized', 'predicted_energy']
    if not df_merged.empty:
        print("Sample Daily Data with Predictions:")
        valid_cols_to_show = [col for col in columns_to_show_sample if col in df_merged.columns]
        print(df_merged[valid_cols_to_show].head())
        print("\nPrediction statistics:")
        if 'predicted_energy' in df_merged.columns and pd.api.types.is_numeric_dtype(df_merged['predicted_energy']):
            if df_merged['predicted_energy'].notna().any(): print("Predicted Energy (Fuzzy Score):\n", df_merged['predicted_energy'].describe())
            else: print("Predicted energy column contains only NaN values.")
        else: print("Cannot describe predicted_energy column.")
        if 'actual_energy_normalized' in df_merged.columns and pd.api.types.is_numeric_dtype(df_merged['actual_energy_normalized']):
            if df_merged['actual_energy_normalized'].notna().any(): print("\nActual Avg Energy (Normalized):\n", df_merged['actual_energy_normalized'].describe())
            else: print("Actual energy normalized column contains only NaN values.")
    else: print("DataFrame is empty, cannot show results.")
    if not df_merged.empty:
        try:
            final_columns = df_merged.columns.tolist()
            print(f"\nSaving final DataFrame with columns: {final_columns}")
            df_merged.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.6f')
            print(f"\nDaily fuzzy predictions generated and saved to '{OUTPUT_CSV_PATH}'.")
        except Exception as e: print(f"\nError saving results to CSV: {e}")
    else: print("\nSkipping saving results as the DataFrame is empty.")

# --- Main Execution ---
if __name__ == "__main__":
    # 1. Load Data
    df_daily_dataset, df_bank_holidays, df_info_households, df_weather = load_data()

    # 3. Handle Missing Values
    df_daily_dataset, df_weather = handle_missing_values(df_daily_dataset, df_weather)

    # 4. Process Dates and Holidays
    df_daily_dataset, df_weather, df_bank_holidays = process_dates_holidays(df_daily_dataset, df_weather, df_bank_holidays)

    # 5. Aggregate Energy and Merge Datasets (Daily Level)
    df_merged = aggregate_and_merge(df_daily_dataset, df_weather, df_bank_holidays)

    # 6. Normalize Features (Weather and Actual Energy + Create Lagged)
    df_merged, normalized_weather_cols = normalize_features(df_merged)

    # 7. Define Fuzzy Variables and MFs (incl. time and lagged energy)
    antecedents, predictedEnergy = define_fuzzy_variables_and_mfs()

    # 7.5 Visualize MFs
    visualize_membership_functions(antecedents, predictedEnergy)

    # *** NEW: 7.6 Visualize Data Distributions ***
    visualize_data_distributions(df_merged, normalized_weather_cols)

    # 8. Generate Rules from Decision Tree
    df_labels = fuzzify_to_labels(df_merged, antecedents, predictedEnergy)
    tree_rules_text = generate_rules_from_tree(df_labels, antecedents) # Generate and print tree rules

    # 9. Define Fuzzy Rules (Manually translated/inspired by tree)
    # *** IMPORTANT: Review the printed tree_rules_text and update this function ***
    rules, antecedents_in_rules = define_fuzzy_rules(antecedents, predictedEnergy)

    # 10. Create Fuzzy Simulation
    energy_simulation = create_fuzzy_simulation(rules, predictedEnergy)

    # 11. Run Predictions
    df_merged = run_predictions(df_merged, energy_simulation, antecedents, predictedEnergy, antecedents_in_rules)

    # 12. Perform Analysis
    df_merged = perform_analysis(df_merged)

    # 13. Inspect and Save Results
    inspect_and_save(df_merged)

    print("\nScript finished.")



Loading datasets...
Looking for daily energy files matching: .\daily_dataset\block_*.csv
Found 112 daily energy block files. Loading and concatenating...


Loading block files:   0%|          | 0/112 [00:00<?, ?it/s]

Combined daily dataset loaded successfully (3510433 rows).
Core datasets loaded successfully.

Handling missing values...
Dropped 30 rows with missing energy data from original daily data.
Dropped 1 rows with missing weather data.

Processing dates and holidays...
Date conversions complete.

Aggregating daily energy and merging datasets...
Calculating daily average energy per household...


C:\Users\mrkun\AppData\Local\Temp\ipykernel_14068\196581226.py:130: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_daily_agg['avg_energy_per_household'].replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\mrkun\AppData\Local\Temp\ipykernel_14068\196581226.py:203: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are settin

Daily average energy calculated for 829 days.
Merging daily aggregated energy with weather and holidays...
Rows after merging aggregated energy with weather: 826
Rows after adding holidays: 826
Added 'month' column.

Total rows in final merged DataFrame (Daily Level): 826
Columns: ['day', 'total_energy_sum', 'household_count', 'avg_energy_per_household', 'temperatureMax', 'temperatureMin', 'dewPoint', 'humidity', 'cloudCover', 'windSpeed', 'pressure', 'is_holiday', 'month']

Normalizing features...
Weather columns to normalize: ['temperatureMax', 'temperatureMin', 'dewPoint', 'humidity', 'cloudCover', 'windSpeed', 'pressure']
Actual weather columns being normalized: ['temperatureMax', 'temperatureMin', 'dewPoint', 'humidity', 'cloudCover', 'windSpeed', 'pressure']
Weather feature normalization complete.
Normalizing actual average energy per household for comparison...
'avg_energy_per_household' normalized and stored in 'actual_energy_normalized'.
Created 'previous_day_energy_normalized

Applying fuzzy simulation:   0%|          | 0/826 [00:00<?, ?it/s]

Predictions complete.

--- Performing Analysis ---

Correlation between predicted_energy and actual_energy_normalized: 0.8862

Generating scatter plot (Predicted vs Actual)...
Scatter plot saved as scatter_plot_daily_pred_vs_actual.png.

Group Analysis by Month:
       actual_energy_normalized  predicted_energy
month                                            
1                      0.771767          0.636144
2                      0.763975          0.643158
3                      0.726367          0.581058
4                      0.635950          0.465344
5                      0.554649          0.368240
6                      0.520511          0.352446
7                      0.505447          0.361670
8                      0.493985          0.356390
9                      0.543476          0.356838
10                     0.608288          0.405414
11                     0.686529          0.514742
12                     0.754796          0.579785

Generating line plot for daily avera